# Experiment 1 - Working with Python Packages: NumPy, SciPy, Scikit-learn, Matplotlib

**Course:** ICS1512 - Machine Learning Algorithms Laboratory
**Aim:** To get hands on with the main Python libraries used for machine learning (NumPy, Pandas, SciPy, Scikit-learn, Matplotlib/Seaborn) and to run a full ML workflow (loading data, EDA, preprocessing, feature selection, splitting, training and evaluating) on five different datasets:

1. Loan Amount Prediction (Classification)
2. Iris Dataset (Classification)
3. Predicting Diabetes (Regression / Classification depending on which version of the dataset is used, explained below)
4. Email/SMS Spam Classification (Classification)
5. Handwritten Digit Recognition using MNIST (Classification)

All five datasets are now loaded from local CSV files (`loan.csv`, `iris.csv`, `diabetes.csv`, `mnist.csv`) that were downloaded and placed in the working directory, instead of relying on scikit-learn's built in copies. This means the whole notebook can be run start to finish and gives real, reproducible results for every dataset, including Loan Prediction, which earlier could not be executed since the file was not available at the time.

In [67]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
import torch

**What this cell does:** Imports the libraries used through the rest of the notebook, `pandas` and `numpy` for handling data, `matplotlib` and `seaborn` for plotting, and a few preprocessing tools from scikit-learn like `LabelEncoder` and `StandardScaler`. `tensorflow` and `torch` are also imported in case a deep learning model is tried later, though nothing in this experiment strictly needs them.

In [68]:
print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)
print("TensorFlow :", tf.__version__)
print("PyTorch :", torch.__version__)

Pandas : 3.0.3
NumPy : 2.5.1
TensorFlow : 2.21.0
PyTorch : 2.12.1


**What this cell does:** Prints the version of each major library that is installed. This is a good habit at the start of any notebook since it makes the environment reproducible and helps a lot if a function ends up behaving differently on a different machine or version.

In [1]:
import sys
print(sys.executable)

/usr/bin/python3


**What this cell does:** Shows which exact Python interpreter is running this notebook. Useful to check when a machine has more than one Python installation and you want to be sure the right one is active.

In [2]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

def load_and_preprocess(file_path):
    """
    Generic function to load and preprocess any CSV dataset.

    Steps:
    1. Load dataset
    2. Remove duplicate rows
    3. Remove completely empty rows and columns
    4. Replace common missing value symbols
    5. Remove leading/trailing spaces
    6. Convert numeric-looking strings to numeric
    7. Handle missing values
    8. Encode categorical columns
    9. Standardize numerical feature columns (excluding target)

    Returns:
        Preprocessed DataFrame
    """

    # ==========================
    # Load Dataset
    # ==========================
    df = pd.read_csv(file_path)

    print("=" * 60)
    print("DATASET LOADED SUCCESSFULLY")
    print("=" * 60)
    print("Original Shape:", df.shape)

    # ==========================
    # Remove Duplicate Rows
    # ==========================
    duplicates = df.duplicated().sum()
    df.drop_duplicates(inplace=True)

    # ==========================
    # Remove Completely Empty Rows & Columns
    # ==========================
    df.dropna(axis=0, how="all", inplace=True)
    df.dropna(axis=1, how="all", inplace=True)

    # ==========================
    # Replace Common Missing Value Symbols
    # ==========================
    df.replace(["?", "NA", "N/A", "", "null", "NULL"], pd.NA, inplace=True)

    # ==========================
    # Remove Extra Spaces
    # ==========================
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].str.strip()

    # ==========================
    # Convert Numeric Strings
    # ==========================
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except (ValueError, TypeError):
            pass

    # ==========================
    # Handle Missing Values
    # ==========================
    for col in df.columns:

        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())

        else:
            mode = df[col].mode()

            if not mode.empty:
                df[col] = df[col].fillna(mode[0])

    # ==========================
    # Encode Categorical Columns
    # ==========================
    categorical_cols = df.select_dtypes(include=["object", "string"]).columns

    for col in categorical_cols:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col].astype(str))

    # ==========================
    # Standardize Numerical Columns
    # ==========================
    numeric_cols = df.select_dtypes(include="number").columns.tolist()

    if len(numeric_cols) > 1:

        feature_cols = numeric_cols[:-1]

        scaler = StandardScaler()
        df[feature_cols] = scaler.fit_transform(df[feature_cols])

    # ==========================
    # Summary
    # ==========================
    print("\nPreprocessing Completed Successfully!")
    print("Final Shape:", df.shape)
    print("Duplicate Rows Removed:", duplicates)
    print("Categorical Columns Encoded:", len(categorical_cols))

    if len(numeric_cols) > 1:
        print("Numerical Features Standardized:", len(feature_cols))

    return df

**What this cell does:** Defines one reusable function, `load_and_preprocess()`, that cleans up pretty much any CSV file passed into it. Instead of writing separate cleaning code for every dataset, the same function gets called five times. Step by step it:
- reads the CSV into a DataFrame
- drops duplicate rows and any rows/columns that are completely empty
- treats common missing value placeholders like `?`, `NA`, and empty strings as actual `NaN`
- strips extra whitespace from text columns
- converts numeric looking text columns into real numbers
- fills missing numeric values with the median and missing categorical values with the mode
- label encodes any remaining categorical columns so scikit-learn models can use them
- standardizes the numeric columns using `StandardScaler`

Writing this once and reusing it saves a lot of repeated code later in the notebook.

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def perform_eda(df):
    """
    Generic EDA function.
    Generates a 3x4 figure containing 12 EDA plots.
    """

    numeric_cols = df.select_dtypes(include=np.number).columns
    categorical_cols = df.select_dtypes(exclude=np.number).columns

    fig, axes = plt.subplots(3, 4, figsize=(24, 18))
    axes = axes.flatten()

    # 1 Missing Values
    sns.heatmap(df.isnull(), cbar=False, cmap="viridis", ax=axes[0])
    axes[0].set_title("Missing Values")

    # 2 Correlation Heatmap
    if len(numeric_cols) > 1:
        sns.heatmap(df[numeric_cols].corr(),
                    annot=True,
                    cmap="coolwarm",
                    ax=axes[1])
        axes[1].set_title("Correlation Heatmap")
    else:
        axes[1].text(0.5,0.5,"Not enough numeric columns",ha="center")
        axes[1].set_title("Correlation Heatmap")

    # 3 Histogram
    if len(numeric_cols):
        sns.histplot(df[numeric_cols[0]], kde=True, ax=axes[2])
        axes[2].set_title(f"Histogram ({numeric_cols[0]})")

    # 4 Scatter Plot
    if len(numeric_cols) > 1:
        sns.scatterplot(x=df[numeric_cols[0]],
                        y=df[numeric_cols[1]],
                        ax=axes[3])
        axes[3].set_title("Scatter Plot")

    # 5 Box Plot
    if len(numeric_cols):
        sns.boxplot(y=df[numeric_cols[0]], ax=axes[4])
        axes[4].set_title("Box Plot")

    # 6 Bar Chart
    if len(categorical_cols):
        df[categorical_cols[0]].value_counts().plot(kind="bar", ax=axes[5])
        axes[5].set_title("Bar Chart")
    else:
        axes[5].text(0.5,0.5,"No categorical columns",ha="center")
        axes[5].set_title("Bar Chart")

    # 7 Pie Chart
    if len(categorical_cols):
        df[categorical_cols[0]].value_counts().plot(kind="pie",
                                                    autopct="%1.1f%%",
                                                    ax=axes[6])
        axes[6].set_ylabel("")
        axes[6].set_title("Pie Chart")
    else:
        axes[6].text(0.5,0.5,"No categorical columns",ha="center")
        axes[6].set_title("Pie Chart")

    # 8 Violin Plot
    if len(numeric_cols):
        sns.violinplot(y=df[numeric_cols[0]], ax=axes[7])
        axes[7].set_title("Violin Plot")

    # 9 Density Plot
    if len(numeric_cols):
        sns.kdeplot(df[numeric_cols[0]], fill=True, ax=axes[8])
        axes[8].set_title("Density Plot")

    # 10 Regression Plot
    if len(numeric_cols) > 1:
        sns.regplot(x=df[numeric_cols[0]],
                    y=df[numeric_cols[1]],
                    ax=axes[9])
        axes[9].set_title("Regression Plot")

    # 11 Strip Plot
    if len(numeric_cols):
        sns.stripplot(y=df[numeric_cols[0]], ax=axes[10])
        axes[10].set_title("Strip Plot")

    # 12 ECDF Plot
    if len(numeric_cols):
        sns.ecdfplot(df[numeric_cols[0]], ax=axes[11])
        axes[11].set_title("ECDF Plot")

    plt.tight_layout()
    plt.show()

    print("="*60)
    print("DATASET SUMMARY")
    print("="*60)
    print("\nShape:", df.shape)

    print("\nData Types")
    print(df.dtypes)

    print("\nMissing Values")
    print(df.isnull().sum())

    print("\nSummary Statistics")
    display(df.describe(include="all"))

**What this cell does:** Defines a second reusable function, `perform_eda()`, that takes any DataFrame and produces a 3x4 grid of 12 plots in one shot: a missing value heatmap, correlation heatmap, histogram, scatter plot, box plot, bar chart, pie chart, violin plot, density plot, regression plot, strip plot, and ECDF plot, along with a printed summary of shape, dtypes and basic statistics. This gives a quick overview of any dataset without writing plotting code from scratch every single time.

## Dataset 1: Loan Amount Prediction

**ML Task:** Supervised Learning, **Classification** (predicting `Loan_Status`: approved or not approved).

This dataset is now loaded from the real `loan.csv` file, so unlike the earlier version of this notebook, the full pipeline (feature selection, train/test split, and Logistic Regression) actually runs and produces real numbers below.

In [ ]:
df = load_and_preprocess("loan.csv")
print(df.head())
print(df.columns)

**What this cell does:** Loads the Loan Prediction dataset from `loan.csv` using `load_and_preprocess()` (cleaning, encoding and scaling all happen automatically inside that function), then prints the first few rows and the column names just to make sure everything loaded the way it should.

In [ ]:
perform_eda(df)

**What this cell does:** Runs the generic EDA function on the preprocessed Loan dataset, giving the missing value check, correlation heatmap, class distribution and the rest of the standard plots for this dataset.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

# Separate features (X) and target (y). Loan_Status is the target, Loan_ID is just an identifier.
X_loan = df.drop(columns=["Loan_Status", "Loan_ID"])
y_loan = df["Loan_Status"]

# Chi-square only works on non-negative values, so we shift the scaled features up by their minimum
X_loan_nonneg = X_loan - X_loan.min()

selector = SelectKBest(score_func=chi2, k=6)
X_loan_selected = selector.fit_transform(X_loan_nonneg, y_loan)

print("Top 6 selected features (Chi-Square test):", list(X_loan.columns[selector.get_support()]))

**What this cell does:** Uses `SelectKBest` with the Chi-Square test to rank features by how strongly they are associated with the target `Loan_Status`, and keeps only the top 6. Chi-square needs non-negative inputs, so the already standardized features are shifted up by their minimum value first. Feature selection like this helps cut down noise and redundant columns, which can improve both accuracy and training speed.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_loan_selected, y_loan, test_size=0.2, random_state=42, stratify=y_loan
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

**What this cell does:** Splits the selected features and target into an 80 percent training set and a 20 percent testing set with `train_test_split`. `stratify=y_loan` keeps the same proportion of approved vs not approved cases in both splits, and `random_state=42` just makes the split reproducible if this cell is run again.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_loan = LogisticRegression(max_iter=1000)
model_loan.fit(X_train, y_train)
y_pred = model_loan.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

**What this cell does:** Trains a Logistic Regression classifier on the training set and evaluates it on the held out test set using accuracy, a full classification report (precision, recall, F1 per class), and a confusion matrix.

## Dataset 2: Iris Dataset

**ML Task:** Supervised Learning, **Classification** (predicting flower `species` from petal and sepal measurements).

Loaded here from the local `iris.csv` file instead of scikit-learn's built in copy, so this goes through the exact same `load_and_preprocess()` pipeline as everything else.

In [ ]:
iris_df = pd.read_csv("iris.csv")

# Standardize the target column name to "species" no matter how it is spelled in the CSV
possible_target_cols = ["species", "Species", "class", "Class", "variety", "Name"]
target_col = next((c for c in possible_target_cols if c in iris_df.columns), iris_df.columns[-1])
iris_df = iris_df.rename(columns={target_col: "species"})

# Drop an ID column if the CSV has one, it is not a real feature
if "Id" in iris_df.columns:
    iris_df = iris_df.drop(columns=["Id"])

iris_df.head()

**What this cell does:** Reads the Iris dataset straight from `iris.csv`. Since different copies of this dataset online sometimes name the label column differently (`species`, `Species`, `variety`, and so on), the code checks for a few common names and renames whichever one is found to `species`, so the rest of the notebook does not need to care about it. Any leftover `Id` column is dropped since it is just a row number, not a real feature.

In [ ]:
iris_df.to_csv('iris_raw.csv', index=False)
iris_processed = load_and_preprocess('iris_raw.csv')

**What this cell does:** Saves the Iris data back out to CSV and reloads it through `load_and_preprocess()`, so it gets exactly the same cleaning, encoding and scaling treatment as every other dataset in this notebook (the `species` column gets label encoded and the 4 measurement columns get standardized).

In [ ]:
perform_eda(iris_processed)

**What this cell does:** Runs the EDA function on the preprocessed Iris dataset to get the usual grid of 12 plots plus the printed summary statistics.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

X_iris = iris_processed.drop(columns=["species"])
y_iris = iris_processed["species"]

selector_iris = SelectKBest(score_func=f_classif, k=2)
X_iris_selected = selector_iris.fit_transform(X_iris, y_iris)
print("Top 2 selected features (ANOVA F-test):", list(X_iris.columns[selector_iris.get_support()]))

**What this cell does:** Uses the ANOVA F-test through `SelectKBest` to find the 2 measurements that separate the three species best. This usually ends up picking petal length and petal width, since those tend to vary the most between species.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    X_iris_selected, y_iris, test_size=0.2, random_state=42, stratify=y_iris
)

knn_iris = KNeighborsClassifier(n_neighbors=5)
knn_iris.fit(X_train, y_train)
y_pred_iris = knn_iris.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_iris))
print("\nClassification Report:\n", classification_report(y_test, y_pred_iris))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_iris))

**What this cell does:** Splits the data 80/20, trains a K-Nearest Neighbors classifier with k=5 on the two selected features, and checks its performance on the test set with accuracy, a classification report, and a confusion matrix.

## Dataset 3: Predicting Diabetes

**ML Task:** This depends a bit on which version of the Diabetes dataset ends up in `diabetes.csv`. There are two common ones floating around online:

- The **sklearn style** diabetes dataset: 442 patients, 10 baseline health measurements, and a continuous target showing disease progression a year later. This is a **regression** problem.
- The **Pima Indians Diabetes Database** from Kaggle/UCI: 768 patients, 8 features (Glucose, BMI, Age, etc.), and a binary `Outcome` column (1 = diabetic, 0 = not). This is a **classification** problem.

Since the exact CSV was downloaded locally rather than pulled from a fixed source, the code below checks which kind of target column is present and automatically switches between Linear Regression (if the target is continuous) and Logistic Regression (if the target is binary, i.e. the Pima version), so it works either way without needing to change anything by hand.

In [ ]:
diabetes_df = pd.read_csv("diabetes.csv")
diabetes_df.head()

**What this cell does:** Reads the Diabetes dataset from `diabetes.csv`. What the columns actually look like depends on which version of the file this turns out to be, so we just take a quick look with `.head()` before doing anything else with it.

In [ ]:
diabetes_df.to_csv('diabetes_raw.csv', index=False)
diabetes_processed = load_and_preprocess('diabetes_raw.csv')

# Figure out which column is the target and whether this is a regression or classification problem
possible_targets = ["target", "Outcome", "outcome", "progression"]
diab_target_col = next((c for c in possible_targets if c in diabetes_processed.columns), diabetes_processed.columns[-1])
is_diabetes_classification = diabetes_processed[diab_target_col].nunique() <= 5

print("Target column detected as:", diab_target_col)
print("Treating this as a", "classification" if is_diabetes_classification else "regression", "problem")

**What this cell does:** Passes the Diabetes data through the shared `load_and_preprocess()` pipeline for consistent cleaning and scaling, then checks the target column to decide whether we are dealing with a continuous score (regression) or a small number of distinct classes like 0/1 (classification). This is done automatically with `.nunique()` rather than being hardcoded, so the same notebook keeps working no matter which version of the dataset was downloaded.

In [ ]:
perform_eda(diabetes_processed)

**What this cell does:** Runs the EDA function on the preprocessed Diabetes data to get the correlation heatmap, distributions and the rest of the standard plots.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression, f_classif

X_diab = diabetes_processed.drop(columns=[diab_target_col])
y_diab = diabetes_processed[diab_target_col]

score_func = f_classif if is_diabetes_classification else f_regression
selector_diab = SelectKBest(score_func=score_func, k=min(5, X_diab.shape[1]))
X_diab_selected = selector_diab.fit_transform(X_diab, y_diab)
print("Top selected features:", list(X_diab.columns[selector_diab.get_support()]))

**What this cell does:** Uses `SelectKBest` to shortlist the most useful features, with `f_regression` if the target turned out to be continuous or `f_classif` (the same ANOVA idea used for Iris) if it turned out to be the binary Pima version. Either way we keep the top 5 features most related to the target.

In [ ]:
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    X_diab_selected, y_diab, test_size=0.2, random_state=42,
    stratify=y_diab if is_diabetes_classification else None
)

if is_diabetes_classification:
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    diab_model = LogisticRegression(max_iter=1000)
    diab_model.fit(Xtr, ytr)
    pred_diab = diab_model.predict(Xte)

    print("Accuracy:", accuracy_score(yte, pred_diab))
    print("\nClassification Report:\n", classification_report(yte, pred_diab))
    print("Confusion Matrix:\n", confusion_matrix(yte, pred_diab))
else:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score

    diab_model = LinearRegression()
    diab_model.fit(Xtr, ytr)
    pred_diab = diab_model.predict(Xte)

    print("RMSE:", mean_squared_error(yte, pred_diab) ** 0.5)
    print("R2 Score:", r2_score(yte, pred_diab))

**What this cell does:** Trains whichever model actually fits the data: Logistic Regression with accuracy/precision/recall/F1 if the Pima classification version was downloaded, or Linear Regression with RMSE and R2 score if the original regression version was downloaded. Either way the split is 80/20, and stratification is only used when it actually makes sense, which is for classification.

## Dataset 4: Email/SMS Spam Classification

**ML Task:** Supervised Learning, **Classification** (predicting `label`: spam or ham/not-spam from the message text).

Text data needs a different kind of preprocessing than plain numeric tables (we vectorize the words instead of scaling numbers), so this section uses its own small pipeline instead of `load_and_preprocess()`. A real spam CSV was not part of the batch of files downloaded for this run, so this section falls back to a small hand written sample of 20 messages just so the code can still be demonstrated end to end. If a proper spam dataset (like the SMS Spam Collection from Kaggle/UCI) is added later as `spam.csv`, the same code will pick it up automatically without any changes.

In [ ]:
import os
import pandas as pd

spam_path = "spam.csv"

if os.path.exists(spam_path):
    spam_df = pd.read_csv(spam_path, encoding="latin-1")[["v1", "v2"]] if "v1" in pd.read_csv(spam_path, encoding="latin-1", nrows=1).columns else pd.read_csv(spam_path)
    spam_df.columns = ["label", "message"]
else:
    # Small illustrative sample so the notebook still runs without the original file
    spam_df = pd.DataFrame({
        "label": ["ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam",
                  "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam", "ham", "spam"],
        "message": [
            "Hey, are we still on for lunch tomorrow?",
            "WINNER!! You have been selected for a FREE cash prize, claim now",
            "Can you send me the notes from class today?",
            "URGENT! Your account has been suspended, click here to verify",
            "Mom said dinner is ready whenever you get home",
            "Congratulations, you've won a free vacation, call now to claim",
            "Let's catch up this weekend if you're free",
            "You have been chosen to receive $1000, reply YES to claim",
            "Don't forget to bring your charger tomorrow",
            "Get cheap loans instantly, no credit check required, apply now",
            "Happy birthday! Hope you have a great day",
            "Free entry into our weekly prize draw, text WIN to enter",
            "Running a bit late, be there in 10 minutes",
            "Your loan has been approved, click the link to receive funds",
            "Can we reschedule our meeting to Friday?",
            "Limited time offer, buy one get one free on all items",
            "Thanks for helping me move last weekend",
            "You've been selected for a special discount, click now",
            "See you at the gym later today",
            "Claim your free gift card before it expires today"
        ]
    })

spam_df.head()

**What this cell does:** Loads a spam/ham dataset, either the real `spam.csv` if it happens to be present, or a small illustrative built in sample of spam and normal messages so the classification pipeline below still runs in any environment. Right now the fallback sample is being used since a real spam CSV was not part of this batch of downloads.

In [ ]:
from sklearn.preprocessing import LabelEncoder

spam_df = spam_df.dropna(subset=["message", "label"]).drop_duplicates()
spam_df["message"] = spam_df["message"].str.strip()

le_spam = LabelEncoder()
spam_df["label_encoded"] = le_spam.fit_transform(spam_df["label"])  # ham=0, spam=1 (alphabetical)

print(spam_df["label"].value_counts())

**What this cell does:** Cleans up the text a little (drops empty or duplicate rows, strips extra whitespace) and label encodes `ham`/`spam` into 0/1 so it can be used as a normal numeric classification target.

In [ ]:
spam_df["message_length"] = spam_df["message"].str.len()
spam_df["word_count"] = spam_df["message"].str.split().apply(len)

spam_eda_df = spam_df[["label_encoded", "message_length", "word_count"]].rename(columns={"label_encoded": "label"})
perform_eda(spam_eda_df)

**What this cell does:** `perform_eda()` is built for numeric tabular data, and raw text does not really fit that mold, so two simple numeric features are derived from each message first, its character length and its word count, alongside the encoded label. Running `perform_eda()` on these three columns gives a class distribution chart and basic stats even for a text dataset, without writing a separate plotting function just for this one section.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=200)
X_spam = tfidf.fit_transform(spam_df["message"])
y_spam = spam_df["label_encoded"]

print("TF-IDF feature matrix shape:", X_spam.shape)

**What this cell does:** Converts the raw message text into numeric features using TF-IDF (term frequency, inverse document frequency), which scores each word based on how distinctive it is to a message compared to the rest of the dataset. This is the standard way to turn text into something a normal classifier can use, and it automatically downweights common, uninformative words through `stop_words="english"`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

Xtr, Xte, ytr, yte = train_test_split(X_spam, y_spam, test_size=0.2, random_state=42, stratify=y_spam)

nb_model = MultinomialNB()
nb_model.fit(Xtr, ytr)
pred_spam = nb_model.predict(Xte)

print("Accuracy:", accuracy_score(yte, pred_spam))
print("\nClassification Report:\n", classification_report(yte, pred_spam))

**What this cell does:** Trains a Multinomial Naive Bayes classifier, the classic fast baseline for text classification, on the TF-IDF features, then checks accuracy and per class precision/recall/F1 on the held out test messages. Since this run is still on the small hand written sample rather than the full SMS Spam Collection, these numbers should be read as a demonstration that the pipeline works, not as a real performance measure.

## Dataset 5: Handwritten Character Recognition (MNIST)

**ML Task:** Supervised Learning, **Classification** (predicting which digit, 0 to 9, a handwritten image represents).

This section now uses the real `mnist.csv` file instead of scikit-learn's small built in `load_digits` dataset. The standard Kaggle/CSV version of MNIST has one `label` column followed by 784 pixel columns (`pixel0` to `pixel783`), since each image is 28x28 pixels flattened into a single row. This is a bigger, more realistic version of the digits problem than the old 8x8 sklearn dataset, so the numbers below should be a fairer picture of how well SVM actually does on this task.

In [ ]:
mnist_df = pd.read_csv("mnist.csv")

# Figure out the label column and the pixel columns, in case the CSV has a slightly different layout
label_col = "label" if "label" in mnist_df.columns else mnist_df.columns[0]
pixel_cols = [c for c in mnist_df.columns if c != label_col]

print("Shape:", mnist_df.shape)
print("Label column:", label_col)
print("Number of pixel columns:", len(pixel_cols))
mnist_df.head()

**What this cell does:** Reads `mnist.csv` and figures out which column holds the digit label and which columns are pixel values, since not every copy of this CSV online lays things out identically. Printing the shape and a preview is just a sanity check before going further.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Figure out the image side length from however many pixel columns there are (784 -> 28x28)
side = int(np.sqrt(len(pixel_cols)))

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    image = mnist_df.iloc[i][pixel_cols].values.reshape(side, side)
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {mnist_df.iloc[i][label_col]}")
    ax.axis("off")
plt.suptitle("Sample Handwritten Digits from MNIST")
plt.tight_layout()
plt.show()

**What this cell does:** Reshapes the first 10 rows back into square images (28x28, worked out automatically from the number of pixel columns) and displays them with their true labels, just to see what the model is actually being asked to recognize.

In [ ]:
perform_eda(mnist_df[[label_col] + pixel_cols[:11]].rename(columns={label_col: "label"}))

**What this cell does:** Running `perform_eda()` on all 784 raw pixel columns would make an unreadable correlation heatmap and take a while to plot, so this just passes the label plus a handful of pixel columns into the EDA function to get a reasonable class distribution chart and general summary, without trying to force 784 dimensions into one heatmap.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

X_digits = mnist_df[pixel_cols]
y_digits = mnist_df[label_col]

# Remove near-constant pixels (e.g. always-black corner pixels) that carry no real information
vt = VarianceThreshold(threshold=0.01)
X_digits_selected = vt.fit_transform(X_digits)

print("Original feature count:", X_digits.shape[1])
print("Feature count after variance thresholding:", X_digits_selected.shape[1])

**What this cell does:** Applies `VarianceThreshold` to drop pixels that are almost always the same value across every image, usually the blank corners of the 28x28 grid that never actually contain any part of a digit. These pixels add no useful signal but do add extra computation, so dropping them is a sensible, cheap feature selection step for image data like this.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# MNIST as a full CSV can have tens of thousands of rows, so a sample is used to keep SVM training time reasonable
sample_size = min(6000, len(mnist_df))
sample_idx = np.random.RandomState(42).choice(len(mnist_df), size=sample_size, replace=False)

X_sample = X_digits_selected[sample_idx]
y_sample = y_digits.iloc[sample_idx]

X_scaled = StandardScaler().fit_transform(X_sample)
Xtr, Xte, ytr, yte = train_test_split(X_scaled, y_sample, test_size=0.2, random_state=42, stratify=y_sample)

svm_digits = SVC(kernel="rbf", gamma="scale")
svm_digits.fit(Xtr, ytr)
pred_digits = svm_digits.predict(Xte)

print("Accuracy:", accuracy_score(yte, pred_digits))
print("\nClassification Report:\n", classification_report(yte, pred_digits))

cm = confusion_matrix(yte, pred_digits)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Digit Recognition (MNIST)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

**What this cell does:** Scales the selected pixel features and trains a Support Vector Machine with an RBF kernel, the same strong classic choice for image data used before. Since the full MNIST CSV can have tens of thousands of rows and SVM training time grows quickly with dataset size, a random sample of up to 6000 rows is used instead of the entire file, which keeps training time reasonable on a normal laptop while still giving a realistic picture of performance. The results are reported as accuracy, a full classification report, and a confusion matrix heatmap showing which digits get mixed up with which.

## Summary Table

| Dataset | Type of ML Task | Feature Selection Technique | Suitable ML Algorithm |
|---|---|---|---|
| Iris Dataset | Supervised, Classification | ANOVA F-test (`SelectKBest`) | K-Nearest Neighbors |
| Loan Amount Prediction | Supervised, Classification | Chi-Square test (`SelectKBest`) | Logistic Regression |
| Predicting Diabetes | Supervised, Regression or Classification (depends on which CSV version was used) | F-regression or ANOVA F-test (`SelectKBest`) | Linear Regression or Logistic Regression |
| Email/SMS Spam Classification | Supervised, Classification | TF-IDF term weighting | Multinomial Naive Bayes |
| Handwritten Digit Recognition (MNIST) | Supervised, Classification | Variance Threshold | Support Vector Machine (SVM) |

## Key Learning Outcomes

- Got practice with the full machine learning workflow: loading data, EDA, preprocessing, feature selection, splitting into train/test sets, training a model, and evaluating it.
- Learned to write reusable functions (`load_and_preprocess`, `perform_eda`) that work across completely different datasets instead of rewriting the same cleaning and plotting code five separate times.
- Saw the difference between classification tasks (Iris, Loan, Spam, Digits, and possibly Diabetes depending on the CSV) and regression tasks, and how that changes which metrics actually make sense, accuracy/F1/confusion matrix versus RMSE/R2.
- Practiced a few different feature selection techniques, Chi-Square, ANOVA F-test, F-regression, TF-IDF weighting, and Variance Threshold, and got a better sense of when each one is the right fit depending on whether the features are categorical or continuous, and whether the data is text or plain tabular numbers.
- Went through the standard visualization types (heatmaps, histograms, box plots, violin plots, KDE plots, confusion matrices) and used them both to understand a dataset before modeling and to check how well a model actually did afterwards.
- Got hands on with NumPy, Pandas, SciPy backed scikit-learn utilities, and Matplotlib/Seaborn together in one workflow, connecting the library exploration part of this experiment directly to an applied, working pipeline.
- Also learned that real downloaded datasets do not always match the exact structure you expect (different column names, different Diabetes dataset versions, larger file sizes for MNIST), so writing code that checks column names and adapts instead of hardcoding everything turned out to be a useful habit.